# Generation Inspection

Explore prompt building and LLM-based answering before or alongside `src/generation/`.

**Goals:**
- Retrieve chunks for a question and inspect `build_qa_prompt`
- Call the LLM with RAG (question + context) and without (question only)
- Compare answers and decide on prompt wording, model, and when to say "I don't know"

**Requires:** An LLM. Use OpenAI (paid), or a free option: Ollama, Groq, or OpenRouter. Set in `.env` — see `.env.example`.

## 1. Setup

### If the LLM doesn't work

- **OpenAI:** One line `OPENAI_API_KEY=sk-proj-...` (no quotes). Requires a paid account.
- **Free options:** Ollama (local), Groq, OpenRouter — set `OPENAI_BASE_URL`, `OPENAI_API_KEY`, and `LLM_MODEL` as in `.env.example`.
- Use one line per variable, no quotes, no space around `=`.  
  Run the cell below to diagnose.

In [1]:
# Diagnose .env — does NOT print your key
import os
import sys
from pathlib import Path
_r = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(_r))
from src.config.settings import PROJECT_ROOT

_env_path = PROJECT_ROOT / ".env"
if _env_path.exists():
    os.environ.pop("OPENAI_API_KEY", None)
    os.environ.pop("OPENAI_BASE_URL", None)
    os.environ.pop("LLM_MODEL", None)
    from dotenv import load_dotenv
    load_dotenv(_env_path, override=True)
    base = os.getenv("OPENAI_BASE_URL") or "(not set)"
    key = os.getenv("OPENAI_API_KEY") or ""
    model = os.getenv("LLM_MODEL") or "(default)"
    print(f"OPENAI_BASE_URL: {base}")
    print(f"LLM_MODEL: {model}")
    print(f"OPENAI_API_KEY: set={bool(key)}, len={len(key)}")
    if not key and base == "(not set)":
        print("  → Set OPENAI_API_KEY or use Ollama/Groq/OpenRouter (OPENAI_BASE_URL). See .env.example.")
else:
    print(".env not found at", _env_path)

OPENAI_BASE_URL: http://localhost:11434/v1
LLM_MODEL: llama3.2
OPENAI_API_KEY: set=True, len=57


In [2]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from src.config.settings import PROJECT_ROOT, CHUNKS_JSONL, INDEX_PATH, TOP_K, LLM_MODEL

_base = os.getenv("OPENAI_BASE_URL")
_key = (os.getenv("OPENAI_API_KEY") or "").strip()
_model = os.getenv("LLM_MODEL") or LLM_MODEL

print(f"LLM: {_model}" + (f" (via {_base})" if _base else " (OpenAI)"))
print(f"Index: {INDEX_PATH}, exists: {INDEX_PATH.exists()}")
print(f"Chunks: {CHUNKS_JSONL}, exists: {CHUNKS_JSONL.exists()}")
print(f"TOP_K: {TOP_K}")
if not INDEX_PATH.exists() and CHUNKS_JSONL.exists():
    print("(Will build in-memory index from chunks.jsonl. Run: python scripts/build_index.py)")

if not _base and not _key:
    print("(No LLM configured. Copy .env.example to .env and choose OpenAI, Ollama, Groq, or OpenRouter.)")
elif _base:
    pass  # Ollama/Groq/OpenRouter; key can be 'ollama' or provider key
elif ("sk-your" in _key and "here" in _key) or len(_key) < 40:
    print("(OPENAI_API_KEY looks like a placeholder. Use a real key or switch to Ollama/Groq/OpenRouter. See .env.example.)")

LLM: llama3.2 (via http://localhost:11434/v1)
Index: /Users/srujayreddy/Projects/ca-dmv-rag-system/data/embeddings/handbook.index, exists: False
Chunks: /Users/srujayreddy/Projects/ca-dmv-rag-system/data/processed/chunks.jsonl, exists: True
TOP_K: 5
(Will build in-memory index from chunks.jsonl. Run: python scripts/build_index.py)


## 2. Retrieve chunks for a question

In [3]:
import json
from src.retrieval import Retriever, embed_query, embed_texts
from src.retrieval.vector_store import VectorStore

question = "What is the blood alcohol limit for DUI?"

if INDEX_PATH.exists():
    retriever = Retriever(INDEX_PATH)
else:
    # Build in-memory index from chunks.jsonl when handbook.index is missing
    if not CHUNKS_JSONL.exists():
        raise FileNotFoundError(f"Need {CHUNKS_JSONL}. Run: python scripts/build_chunks.py")
    chunks_list = []
    with open(CHUNKS_JSONL) as f:
        for line in f:
            if line.strip():
                chunks_list.append(json.loads(line))
    texts = [c["text"] for c in chunks_list]
    embs = embed_texts(texts)
    _store = VectorStore()
    _store.add(embs, chunks_list)
    class _InMemoryRetriever:
        def retrieve(self, q, top_k=5):
            return _store.search(embed_query(q), top_k=top_k)
    retriever = _InMemoryRetriever()
    print("(Using in-memory index from chunks.jsonl. Run: python scripts/build_index.py to persist handbook.index)\n")

chunks = retriever.retrieve(question, top_k=TOP_K)
print(f"Question: {question}")
print(f"Retrieved {len(chunks)} chunks")
for i, c in enumerate(chunks[:2], 1):
    print(f"  {i}. page {c.get('page')}, score={c.get('score', 0):.4f}")
    print(f"     {c['text'][:120]}...")

/Users/srujayreddy/Projects/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


(Using in-memory index from chunks.jsonl. Run: python scripts/build_index.py to persist handbook.index)

Question: What is the blood alcohol limit for DUI?
Retrieved 5 chunks
  1. page 79, score=0.7315
     73
Blood Alcohol Concentration (BAC) Limits
When you consume alcohol, traces of it enter your bloodstream. Your 
BAC mea...
  2. page 80, score=0.5991
      while driving under the influence 
of drugs or alcohol, you may face civil lawsuits.
All DUI convictions remain on your...


## 3. Inspect the RAG prompt

In [4]:
from src.generation import build_qa_prompt

prompt = build_qa_prompt(question, chunks)
print(f"Prompt length: {len(prompt)} chars")
print("--- First 800 chars ---")
print(prompt[:800])
print("...")

Prompt length: 2209 chars
--- First 800 chars ---
You are a helpful assistant. Answer the question using only the following context from the California DMV Driver Handbook. If the context does not contain the answer, say "I don't know."

Context:

73
Blood Alcohol Concentration (BAC) Limits
When you consume alcohol, traces of it enter your bloodstream. Your 
BAC measures how much alcohol is present in your bloodstream.
It is illegal for you to drive if you have a BAC of:
• 0.08% or higher if you are over 21 years old.
• 0.01% or higher if you are under 21 years old.
• 0.01% or higher at any age if you are on DUI probation.
• 0.04% or higher if you drive a vehicle that requires a commercial 
driver’s license. 
• 0.04% or higher if you are driving a pa
(Page 79)

---

 while driving under the influence 
of drugs or alcohol, you may face civ
...


## 4. Generate answer with RAG

In [5]:
from openai import AuthenticationError, NotFoundError, RateLimitError
from src.generation import answer

try:
    out = answer(question, context_chunks=chunks)
    print(f"Question: {question}")
    print(f"Answer (RAG):\n{out}")
except ValueError as e:
    print(f"Error: {e}")
    print("Configure an LLM in .env: OpenAI, or Ollama/Groq/OpenRouter. See .env.example.")
except AuthenticationError:
    print("Invalid API key. Use a valid key or Ollama/Groq/OpenRouter. See .env.example.")
except NotFoundError:
    print("Model not found. For Ollama: run  ollama list  to see models; if empty, run  ollama pull llama3.2 . Then set LLM_MODEL in .env to that exact name (e.g. llama3.2).")
except RateLimitError:
    print("Quota exceeded (429). Use a paid OpenAI plan or Ollama/Groq/OpenRouter. See .env.example.")

Question: What is the blood alcohol limit for DUI?
Answer (RAG):
According to the California DMV Driver Handbook, the blood alcohol limits for DUI are:

* 0.08% or higher if you are over 21 years old
* 0.01% or higher if you are under 21 years old
* 0.01% or higher at any age if you are on DUI probation
* 0.04% or higher if you drive a vehicle that requires a commercial driver's license


## 5. Compare: with vs without RAG

In [6]:
# Without RAG: model answers from general knowledge (may differ from handbook)
from openai import AuthenticationError, NotFoundError, RateLimitError
try:
    out_no_rag = answer(question, context_chunks=None)
    print("Answer (no RAG, general knowledge):")
    print(out_no_rag)
except ValueError as e:
    print(f"Error: {e}")
except AuthenticationError:
    print("Invalid API key. Use a valid key or Ollama/Groq/OpenRouter. See .env.example.")
except NotFoundError:
    print("Model not found. For Ollama: run  ollama list  to see models; if empty, run  ollama pull llama3.2 . Then set LLM_MODEL in .env to that exact name (e.g. llama3.2).")
except RateLimitError:
    print("Quota exceeded (429). Use paid OpenAI or Ollama/Groq/OpenRouter. See .env.example.")

Answer (no RAG, general knowledge):
In the United States, the blood alcohol concentration (BAC) limits for driving under the influence (DUI) vary from state to state. However, here are some general guidelines:

* For most states, a BAC of 0.08% or higher is considered impaired and can result in a DUI charge.
* In some states, a BAC of 0.05% to 0.07% may also be considered impaired, especially for commercial drivers or those operating vehicles with children on board.
* For commercial drivers, the Federal Motor Carrier Safety Administration (FMCSA) sets a zero-tolerance policy for BAC, meaning any amount above 0% is considered impaired.

It's essential to note that these limits can vary significantly from state to state, and some states have even lower or higher limits. It's always best to check your state's specific laws and regulations regarding DUI and BAC limits.

Here are some examples of BAC limits by state:

* California: 0.08%
* Texas: 0.08%
* Florida: 0.08%
* New York: 0.05% (fo

## 6. Try more questions

In [7]:
from openai import NotFoundError, RateLimitError

questions = [
    "When must I use headlights?",
    "How do I renew my driver license?",
    "What is the speed limit on a highway?",
]

for q in questions:
    ch = retriever.retrieve(q, top_k=TOP_K)
    try:
        a = answer(q, context_chunks=ch)
        print(f"Q: {q}")
        print(f"A: {a[:200]}..." if len(a) > 200 else f"A: {a}")
        print()
    except ValueError:
        print(f"Q: {q} (skip: no API key)")
        print()
    except NotFoundError:
        print(f"Q: {q} (skip: model not found; set LLM_MODEL in .env, e.g. llama3.2 for Ollama)")
        print()
    except RateLimitError:
        print(f"Q: {q} (skip: quota exceeded; use paid OpenAI or Ollama/Groq/OpenRouter. See .env.example)")
        print()

Q: When must I use headlights?
A: According to the California DMV Driver Handbook, you must use your headlights:

* When it is too dark to see from 1,000 feet away
* Beginning 30 minutes after sunset
* Until 30 minutes before sunrise
...

Q: How do I renew my driver license?
A: To renew your driver's license, you may visit the DMV office or use online services such as dmv.ca.gov/dlservices. You will need to provide additional proof of identity and submit a request with your ...

Q: What is the speed limit on a highway?
A: The ideal maximum speed limit on most California highways is 65 mph, unless otherwise posted.



## 7. Summary & next steps

In [8]:
print("Decisions for src/generation:")
print("  - prompts: build_qa_prompt with instructions + context (text + page) + question")
print("  - answer: OpenAI chat.completions, temperature=0; RAG if context_chunks else question-only")
print("  - ask.py: Retriever.retrieve -> answer(question, chunks) -> print")

Decisions for src/generation:
  - prompts: build_qa_prompt with instructions + context (text + page) + question
  - answer: OpenAI chat.completions, temperature=0; RAG if context_chunks else question-only
  - ask.py: Retriever.retrieve -> answer(question, chunks) -> print
